In [4]:
# ============================================================
# CELDA 1 — Dependencias
# ============================================================
!pip install gradio --quiet
!pip install nltk --quiet

# ============================================================
# CELDA 2 — Imports y configuración
# ============================================================
import re
import nltk
from collections import defaultdict
import gradio as gr

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# CELDA 3 — Cargar corpus desde el .txt
# ============================================================
CORPUS_PATH = "/content/drive/MyDrive/upslp/corpus_tecnologia.txt"

def cargar_corpus(path):
    corpus = {}
    tema_actual = None
    try:
        with open(path, "r", encoding="utf-8") as f:
            for linea in f:
                linea = linea.strip()
                if not linea:
                    continue
                if linea.startswith("[") and linea.endswith("]"):
                    tema_actual = linea[1:-1].lower()
                    corpus[tema_actual] = []
                elif tema_actual:
                    corpus[tema_actual].append(linea)
        total = sum(len(v) for v in corpus.values())
        print(f"✓ Corpus cargado: {total} oraciones en {len(corpus)} temas")
    except FileNotFoundError:
        print("⚠ No se encontró el archivo del corpus")
    return corpus

CORPUS = cargar_corpus(CORPUS_PATH)

# ============================================================
# CELDA 4 — Palabras clave por tema
# ============================================================
PALABRAS_CLAVE = {
    "virus": [
        "virus", "malware", "troyano", "ransomware", "spyware",
        "antivirus", "gusano", "infección", "infectar", "adware",
        "rootkit", "botnet", "keylogger", "exploit"
    ],
    "ciberseguridad": [
        "ciberseguridad", "seguridad", "ataque", "phishing", "firewall",
        "cifrado", "ddos", "vpn", "contraseña", "hack", "vulnerabilidad",
        "pentesting", "brecha", "intrusión", "parche", "siem", "zero trust"
    ],
    "inteligencia artificial": [
        "inteligencia", "artificial", "ia", "machine", "learning", "deep",
        "neural", "neuronal", "gpt", "chatbot", "nlp", "transformer",
        "generativa", "entrenamiento", "modelo", "algoritmo"
    ],
    "iot": [
        "iot", "internet", "cosas", "sensor", "dispositivo", "wearable",
        "smart", "inteligente", "zigbee", "mqtt", "edge", "gateway",
        "industria", "conectado", "actuador"
    ],
    "redes": [
        "red", "redes", "tcp", "ip", "router", "switch", "wifi", "dns",
        "protocolo", "5g", "lan", "wan", "subred", "vlan", "qos",
        "https", "sdn", "latencia", "ancho", "banda"
    ],
    "computacion en la nube": [
        "nube", "cloud", "aws", "azure", "google cloud", "contenedor",
        "docker", "kubernetes", "serverless", "cdn", "saas", "paas",
        "iaas", "elasticidad", "migración", "híbrida"
    ],
    "computacion cuantica": [
        "cuántica", "cuantica", "qubit", "quantum", "superposición",
        "entrelazamiento", "ibm", "grover", "shor", "criptografía"
    ],
    "programacion": [
        "programación", "programacion", "python", "código", "codigo",
        "algoritmo", "api", "git", "desarrollo", "software", "objeto",
        "clase", "test", "microservicio", "docker", "agil", "sprint"
    ],
}

TODAS_LAS_PALABRAS = {p for lista in PALABRAS_CLAVE.values() for p in lista}

# ============================================================
# CELDA 5 — Motor de búsqueda por similitud
# ============================================================
STOPWORDS_ES = set(stopwords.words('spanish'))

def limpiar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    return [t for t in texto.split() if t not in STOPWORDS_ES and len(t) > 2]

def detectar_tema(tokens):
    scores = defaultdict(int)
    for token in tokens:
        for tema, palabras in PALABRAS_CLAVE.items():
            for palabra in palabras:
                if token == palabra:
                    scores[tema] += 3
                elif token in palabra or palabra in token:
                    scores[tema] += 1
    return max(scores, key=scores.get) if scores else None

def similitud_jaccard(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    if not a or not b:
        return 0
    return len(a & b) / len(a | b)

def buscar_oraciones(tokens_input, tema_detectado, top_n=3):
    candidatas = []
    for tema, oraciones in CORPUS.items():
        bonus = 0.25 if tema == tema_detectado else 0
        for oracion in oraciones:
            score = similitud_jaccard(tokens_input, limpiar(oracion)) + bonus
            if score > 0:
                candidatas.append((score, oracion))
    vistas = set()
    resultado = []
    for score, oracion in sorted(candidatas, reverse=True):
        if oracion not in vistas:
            vistas.add(oracion)
            resultado.append(oracion)
        if len(resultado) >= top_n:
            break
    return resultado

def es_tecnologia(tokens):
    return any(t in TODAS_LAS_PALABRAS for t in tokens)

def generar_respuesta(input_usuario):
    tokens = limpiar(input_usuario)
    if not es_tecnologia(tokens):
        return ("Lo siento, solo respondo sobre tecnología: IA, IoT, "
                "ciberseguridad, redes, computación y temas relacionados.")
    tema = detectar_tema(tokens)
    oraciones = buscar_oraciones(tokens, tema, top_n=3)
    if not oraciones:
        return ("No encontré información específica. Intenta preguntar sobre: "
                "IA, virus, redes, IoT, nube, ciberseguridad o computación cuántica.")
    encabezado = f"Sobre {tema.upper()}:\n\n" if tema else ""
    return encabezado + "\n\n".join(f"• {o}" for o in oraciones)

# ============================================================
# CELDA 6 — Interfaz Gradio
# ============================================================
CSS = """
    .gradio-container {
        max-width: 720px !important;
        margin: auto !important;
        background: #f7f8fc !important;
    }
    body, .gradio-container, .main {
        background: #f7f8fc !important;
    }
    footer { display: none !important; }
    #chat-box { height: 480px; }
    .title-bar {
        background: #1a1a2e;
        color: #e0e0ff;
        padding: 14px 20px;
        border-radius: 10px 10px 0 0;
        display: flex;
        align-items: center;
        gap: 10px;
        font-size: 15px;
        font-weight: 600;
        margin-bottom: 0px;
    }
    .dot {
        width: 10px; height: 10px;
        border-radius: 50%; background: #4ade80;
        display: inline-block;
    }
    .subtitle { font-size: 12px; color: #9999cc; margin-left: auto; }
    #send-btn {
        background: #1a1a2e !important;
        color: #e0e0ff !important;
        border: none !important;
    }
    #send-btn:hover { background: #2d2d50 !important; }
"""

def responder(mensaje, historial):
    if not mensaje.strip():
        return "", historial
    respuesta = generar_respuesta(mensaje)
    historial.append((mensaje, respuesta))
    return "", historial

with gr.Blocks(css=CSS, theme=gr.themes.Default()) as demo:

    gr.HTML("""
        <div class="title-bar">
            <span class="dot"></span>
            <span>TechBot</span>
            <span class="subtitle">IA · Redes · IoT · Ciberseguridad</span>
        </div>
    """)

    chatbot = gr.Chatbot(
        value=[
            (None, "Hola, soy TechBot. Pregúntame sobre IA, virus, IoT, "
                   "redes, ciberseguridad, computación cuántica o programación.")
        ],
        elem_id="chat-box",
        label="",
        show_label=False,
        bubble_full_width=False,
        avatar_images=(None, None),
    )

    with gr.Row():
        txt = gr.Textbox(
            placeholder="Pregunta sobre IA, virus, redes, IoT...",
            scale=8,
            show_label=False,
            container=False,
        )
        btn = gr.Button("Enviar", scale=1, elem_id="send-btn")

    txt.submit(responder, [txt, chatbot], [txt, chatbot])
    btn.click(responder, [txt, chatbot], [txt, chatbot])

demo.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Corpus cargado: 109 oraciones en 8 temas


/tmp/ipykernel_21139/3325432329.py:202: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Default()) as demo:
/tmp/ipykernel_21139/3325432329.py:202: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Default()) as demo:
/tmp/ipykernel_21139/3325432329.py:212: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_21139/3325432329.py:212: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Grad

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2cad5bb1a316753ef2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
